<a href="https://colab.research.google.com/github/weagan/Share-PEFT/blob/main/Standard_LoRA_with_catastrophic_forgetting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Combined from original imports and cell aa0e1302, and get_loader from 86359028
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

## 1. Standard LoRA
class SimpleLoRALinear(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.register_buffer("weight", weight.clone())
        # Initialize lora_A and lora_B on the same device as the weight
        self.lora_A = nn.Parameter(torch.randn(rank, self.in_features, device=weight.device) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=weight.device))
        self.scaling = 0.05

    def forward(self, x):
        delta_w = self.lora_B @ self.lora_A
        return x @ (self.weight + delta_w * self.scaling).T

## 3. Manager
class ContinualSubspaceManager:
    def __init__(self, model_name="distilbert-base-uncased", rank=16):
        # Determine the primary device (cuda:0 if any GPU, else cpu)
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # Load model and move to primary device
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        self.model.to(self.device)

        # Check for multiple GPUs and wrap with DataParallel if available
        self.num_gpus = torch.cuda.device_count()
        if self.num_gpus > 1:
            print(f"Using {self.num_gpus} GPUs with DataParallel.")
            self.model = nn.DataParallel(self.model)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.rank = rank

        # Freeze the base model parameters.
        # If DataParallel is used, freeze the parameters of the underlying module.
        base_model_for_freezing = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in base_model_for_freezing.parameters():
            param.requires_grad = False

    def _get_target_module_and_attr(self, name, model_obj):
        """Helper to get the actual parent module and attribute name to modify."""
        parts = name.split('.')
        current_obj = model_obj
        for part in parts[:-1]:
            current_obj = getattr(current_obj, part)
        return current_obj, parts[-1]

    def inject_standard_lora(self):
        # Work on the base model, whether it's wrapped by DataParallel or not
        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for name, module in list(target_model.named_modules()): # Use list to allow modification during iteration
            if any(tgt_name in name for tgt_name in ["attention.out_lin", "attention.v_lin"]):
                parent, attr_name = self._get_target_module_and_attr(name, target_model)
                setattr(parent, attr_name, SimpleLoRALinear(module.weight, self.rank))

    def train_task(self, task_name, loader, epochs=3):
        print(f"\n🔥 Training {task_name} (Standard LoRA)")
        trainable_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        params = [p for n, p in trainable_model.named_parameters() if "lora_" in n]
        optimizer = optim.AdamW(params, lr=1e-3)

        self.model.train()
        for epoch in range(epochs):
            for batch in tqdm(loader, leave=False):
                optimizer.zero_grad()
                labels = batch.pop("labels").to(self.device)
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**inputs, labels=labels)
                outputs.loss.backward()
                optimizer.step()

    def evaluate_task(self, loader):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in loader:
                labels = batch.pop("labels").to(self.device)
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**inputs)
                correct += (outputs.logits.argmax(-1) == labels).sum().item()
                total += labels.size(0)
        return correct / total

# Helper function to get data loaders
def get_loader(task, tokenizer):
    ds = load_dataset("glue", task)
    def tokenize_fn(ex):
        t = (ex["sentence"],) if "sentence" in ex else (ex["sentence1"], ex["sentence2"])
        res = tokenizer(*t, truncation=True, padding=False)
        res["labels"] = ex["label"]
        return res
    tok = ds.map(tokenize_fn, batched=True, remove_columns=ds["train"].column_names)
    return DataLoader(tok["train"], batch_size=16, shuffle=True, collate_fn=DataCollatorWithPadding(tokenizer)), \
           DataLoader(tok["validation"], batch_size=16, collate_fn=DataCollatorWithPadding(tokenizer))

In [ ]:
# Combined from cells 86359028 and 14518986
manager = ContinualSubspaceManager()
tasks = ["cola", "mrpc", "sst2"]
loaders = {t: get_loader(t, manager.tokenizer) for t in tasks}

baseline_accuracies = {}

print("Evaluating base model on validation sets...")
# Evaluate baseline accuracies before injecting LoRA
for task in tasks:
    _, val_loader = loaders[task]
    acc = manager.evaluate_task(val_loader)
    baseline_accuracies[task] = acc
    print(f"  {task} baseline accuracy: {acc:.3f}")

manager.inject_standard_lora()
print("Standard LoRA modules injected.")

forgetting_table = np.zeros((len(tasks), len(tasks)))

# Sequential training and evaluation for standard LoRA
for i, task in enumerate(tasks):
    # Train current task with standard LoRA
    manager.train_task(task, loaders[task][0])

    # Evaluate on all tasks learned so far
    for j in range(i + 1):
        prev_task = tasks[j]
        # With standard LoRA, the weights are continually updated,
        # so we evaluate the current state of LoRA on all previously learned tasks.
        acc = manager.evaluate_task(loaders[prev_task][1])
        forgetting_table[i, j] = acc

## 5. Result Display
print("\n--- BASELINE ACCURACIES ---")
for task_name, acc in baseline_accuracies.items():
    print(f"  {task_name} baseline accuracy: {acc:.3f}")

print("\n--- FORGETTING TABLE (Accuracy) ---")
# Prepare header for forgetting table
header = " | ".join([f"{t:<6}" for t in tasks])
print(f"{'After Task':<12} | {header}")

for i, row in enumerate(forgetting_table):
    row_values = []
    for j, acc_val in enumerate(row):
        if j <= i: # Only display results for tasks learned up to this point
            row_values.append(f"{acc_val:.3f}")
        else:
            row_values.append("------") # Indicate tasks not yet trained
    print(f"After {tasks[i]:<7} | {' | '.join(row_values)}")

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating base model on validation sets...
  cola baseline accuracy: 0.608
  mrpc baseline accuracy: 0.635
  sst2 baseline accuracy: 0.524
Standard LoRA modules injected.

🔥 Training cola (Standard LoRA)



🔥 Training mrpc (Standard LoRA)



🔥 Training sst2 (Standard LoRA)



--- BASELINE ACCURACIES ---
  cola baseline accuracy: 0.608
  mrpc baseline accuracy: 0.635
  sst2 baseline accuracy: 0.524

--- FORGETTING TABLE (Accuracy) ---
After Task   | cola   | mrpc   | sst2  
After cola    | 0.779 | ------ | ------
After mrpc    | 0.325 | 0.836 | ------
After sst2    | 0.437 | 0.419 | 0.884


In [ ]:
print(manager.device)

cuda:0
